# Pipeline RAG passo a passo

Este notebook executa o pipeline do projeto uma etapa por vez e mostra o resultado intermediário de cada uma. Escrevemos o código de cada etapa aqui, na ordem de execução, para você acompanhar o fluxo sem abrir outros arquivos. No fim, o notebook roda a classe `VoCPipeline` do projeto com a mesma pergunta.

Etapas:

1. Base preparada
2. Documentos
3. Embeddings
4. Índice vetorial
5. Pergunta
6. Análise da pergunta
7. Filtros de metadados
8. Busca semântica
9. Busca híbrida, rerank e seleção das evidências
10. Panorama da base
11. Construção do contexto
12. Geração da resposta
13. Validação das citações
14. Resultado final
15. Mesmo fluxo com as classes do projeto
16. Recusas e calibração do corte de similaridade

Antes de rodar, crie o arquivo `.env` na raiz do projeto com a chave do provedor escolhido (`OPENAI_API_KEY` ou `GOOGLE_API_KEY`) e deixe os CSVs em `data/raw/`. Se a base preparada ou o índice não existirem, o notebook cria os dois. A indexação completa com `text-embedding-3-small` custa alguns centavos de dólar. Cada pergunta faz uma chamada de análise, uma de rerank por frase de busca e uma de geração.

## Configuração

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

from voc_rag.config import Settings
from voc_rag.llm import require_api_key

settings = Settings()
require_api_key(settings.llm_provider)

pd.set_option("display.max_colwidth", 120)
pd.DataFrame(
    [
        ("modelo de chat", f"{settings.llm_provider}:{settings.chat_model}"),
        ("reranker", settings.cross_encoder_model if settings.reranker == "cross_encoder" else settings.rerank_model),
        ("banco vetorial", settings.vector_store),
        ("modelo de embeddings", f"{settings.embedding_provider}:{settings.embedding_model}"),
        ("multi-query, BM25 e rerank", f"{settings.use_multi_query}, {settings.use_bm25}, {settings.use_rerank}"),
        ("candidatos da busca densa e do BM25", f"{settings.fetch_k} e {settings.lexical_k}"),
        ("similaridade mínima", settings.min_similarity),
        ("corte do rerank", settings.min_rerank_score if settings.reranker == "cross_encoder" else settings.min_relevance),
        ("evidências mantidas (específica e panorama)", f"{settings.top_k} e {settings.overview_top_k}"),
        ("evidências mínimas para responder", settings.min_evidences),
    ],
    columns=["parâmetro", "valor"],
)

Os valores vêm de `voc_rag/config.py`. Você troca o provedor, os modelos, a similaridade mínima e os componentes da busca no `.env`.

## 1. Base preparada

A `ReviewDataPreparer` gera duas tabelas. A analítica guarda as 98.410 avaliações e alimenta os cálculos. A de documentos tem só as avaliações com texto útil e alimenta o banco vetorial. O notebook `01_eda_e_montagem_da_base.ipynb` explica cada coluna.

In [ ]:
from voc_rag.preparation import ReviewDataPreparer

if not settings.documents_path.exists():
    print(ReviewDataPreparer(settings).run())

analytics = pd.read_parquet(settings.analytics_path)
documents_df = pd.read_parquet(settings.documents_path)

print(f"Tabela analítica: {len(analytics)} avaliações")
print(f"Tabela de documentos: {len(documents_df)} avaliações indexáveis")
documents_df.head(3)

## 2. Documentos

Cada linha da tabela de documentos vira um `Document` do LangChain. O `page_content` é o texto que recebe o embedding. O Chroma guarda os `metadata` junto do vetor e filtra a busca por eles. O `id` é o `review_id`, que liga a citação à avaliação original.

A função tira os valores nulos dos metadados porque o Chroma rejeita `None`. Os pedidos não entregues têm `delay_days` nulo.

In [ ]:
from langchain_core.documents import Document

METADATA_COLUMNS = [
    "review_id", "score", "category", "delivery_status", "delay_days",
    "purchase_month", "purchase_month_num", "state",
]


def to_document(row: dict) -> Document:
    metadata = {column: row[column] for column in METADATA_COLUMNS if not pd.isna(row[column])}
    return Document(id=row["review_id"], page_content=row["text"], metadata=metadata)


example_document = to_document(documents_df.head(1).to_dict("records")[0])
print("id:", example_document.id)
print("page_content:", example_document.page_content)
example_document.metadata

## 3. Embeddings

Um embedding transforma um texto em um vetor de números. Textos com sentido parecido geram vetores próximos. O modelo `text-embedding-3-small` gera vetores de 1.536 dimensões.

A célula abaixo gera embeddings de três frases e calcula a similaridade de cosseno entre elas, depois de normalizar os vetores para norma 1. Com norma 1, o cosseno é igual ao produto escalar.

In [ ]:
import numpy as np

from voc_rag.llm import create_embeddings

# Com o E5 local, embed_documents acrescenta o prefixo "passage: " e embed_query, o prefixo "query: ".
embeddings = create_embeddings(settings)

sentences = [
    "Não recebi o produto até hoje.",
    "O pedido nunca chegou na minha casa.",
    "Adorei a cor da toalha, muito macia.",
]
vectors = np.array(embeddings.embed_documents(sentences))
vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

print(f"Dimensões de cada vetor: {vectors.shape[1]}")
pd.DataFrame(vectors @ vectors.T, index=sentences, columns=sentences).round(3)

As duas primeiras frases quase não compartilham palavras e descrevem o mesmo problema. Espere uma similaridade entre elas maior do que a de cada uma com a terceira. A busca semântica usa essa proximidade para achar avaliações escritas com outras palavras.

A coleção do Chroma usa distância de cosseno, `distância = 1 - cosseno`, então `cosseno = 1 - distância`. A escolha explícita vale para qualquer provedor: nem todo modelo de embeddings devolve vetores com norma 1.

In [ ]:
cosine_distance = 1 - vectors[0] @ vectors[1]
print(f"Distância de cosseno: {cosine_distance:.4f}")
print(f"1 - distância:        {1 - cosine_distance:.4f}")

## 4. Índice vetorial

A `ReviewIndexer` converte as 39.192 linhas em documentos com a mesma função da etapa 2 e envia lotes de 1.000 ao Chroma. Dentro de `add_documents`, o LangChain chama o provedor de embeddings e o Chroma grava vetores, textos e metadados em `data/chroma/`.

Cada modelo de embeddings tem sua própria coleção, com o nome do modelo. Se a coleção já estiver completa, o notebook reaproveita o índice.

In [ ]:
from voc_rag.indexing import ReviewIndexer, create_vector_store, index_name

try:
    store = create_vector_store(settings)
    outdated = store.count() != len(documents_df)
except RuntimeError:
    outdated = True
if outdated:
    print(f"{ReviewIndexer(settings).build()} documentos indexados")
    store = create_vector_store(settings)

print(f"Índice: {index_name(settings)}")
print(f"Documentos no índice: {store.count()}")

## 5. Pergunta

Troque o texto abaixo para testar outras perguntas. As células seguintes usam a variável `question`.

In [ ]:
question = "Quais são os principais problemas relatados nas avaliações com nota baixa?"

## 6. Análise da pergunta

Uma única chamada ao LLM faz quatro coisas.

- Decide se a pergunta está no escopo da base, no campo `in_scope`.
- Classifica a pergunta como panorama (`overview`), que pede principais temas, ou específica (`specific`).
- Reescreve a pergunta como um cliente escreveria uma avaliação, no campo `search_queries`. Perguntas de panorama recebem de 4 a 6 frases, uma por subtema, e as específicas recebem 1 ou 2. A pergunta "quais os problemas?" usa palavras diferentes das avaliações, que dizem "não chegou" ou "veio quebrado". As frases reescritas ficam mais perto delas no espaço vetorial.
- Extrai filtros de metadados no campo `filters`.

O prompt recebe a lista de categorias existentes para que o modelo use nomes válidos, e também o recorte escolhido na interface, quando houver.

In [ ]:
from voc_rag.prompts import ANALYZER_SYSTEM

print(ANALYZER_SYSTEM)

O modelo responde no formato do esquema Pydantic `QueryAnalysis`. A chamada `with_structured_output` envia esse esquema à API, e o LangChain devolve a resposta como objeto Python validado. Você não precisa interpretar texto livre. A tabela abaixo lista os campos do filtro.

In [ ]:
from voc_rag.schemas import QueryAnalysis, ReviewFilters

pd.DataFrame(
    [(name, str(field.annotation), field.description) for name, field in ReviewFilters.model_fields.items()],
    columns=["campo", "tipo", "descrição"],
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

from voc_rag.llm import create_chat_model

llm = create_chat_model(settings.llm_provider, settings.chat_model, settings.temperature)
categories = sorted(analytics["category"].unique())

analyzer_prompt = ChatPromptTemplate.from_messages([("system", ANALYZER_SYSTEM), ("human", "{question}")])
analyzer_prompt = analyzer_prompt.partial(categories=", ".join(categories))
analyzer_chain = analyzer_prompt | llm.with_structured_output(QueryAnalysis)

analysis = analyzer_chain.invoke({"question": question, "manual_filters": "nenhum"})
if analysis.filters.category not in categories:
    analysis.filters.category = None

analysis.model_dump()

O `if` final descarta categorias inventadas pelo modelo. Um filtro com categoria inexistente zeraria a busca. O `QueryAnalyzer` do projeto também remove frases de busca vazias ou repetidas e limita a lista a seis.

## 7. Filtros de metadados

O método `conditions()` de `ReviewFilters` transforma os campos preenchidos em uma lista de condições. Ele ignora notas fora de 1 a 5, meses em formato inválido e siglas de UF que não existem, e converte meses como `2017-11` para o número `201711`.

A mesma lista serve para montar o filtro do Chroma, para filtrar o BM25 e, na etapa 10, para filtrar a tabela analítica com pandas. Com isso, a busca e o panorama usam o mesmo recorte.

O Chroma exige `$and` quando há mais de uma condição. O filtro de estados usa `$in`.

In [ ]:
from voc_rag.stats import filter_mask

conditions = analysis.filters.conditions()
in_slice = int(filter_mask(documents_df, analysis.filters).sum())

# No FAISS, as condições viram uma máscara e o IDSelectorBatch restringe a busca ao recorte antes de ordenar.
print("Condições:", conditions)
print(f"Documentos no recorte: {in_slice} de {len(documents_df)}")

## 8. Busca semântica

O Chroma gera o embedding da primeira frase de busca, aplica o filtro e devolve os 40 documentos mais próximos, do mais próximo para o mais distante. A função `results_table` converte a distância em similaridade com a fórmula da etapa 3.

In [ ]:
from voc_rag.vector_store import search_with_similarity

search_query = analysis.search_queries[0]
results = search_with_similarity(store, search_query, settings.fetch_k, analysis.filters)


def results_table(search_results) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "posição": position,
                "similaridade": round(similarity, 3),
                "nota": metadata.get("score"),
                "categoria": metadata.get("category"),
                "entrega": metadata.get("delivery_status"),
                "texto": text,
            }
            for position, (text, metadata, similarity) in enumerate(search_results, start=1)
        ],
        columns=["posição", "similaridade", "nota", "categoria", "entrega", "texto"],
    )


print(f"Frase de busca: {search_query}")
results_table(results).head(15)

Para ver o efeito da reescrita, a célula abaixo repete a busca com a pergunta original, sem reescrever. Compare as similaridades e os textos encontrados.

In [ ]:
original_results = search_with_similarity(store, question, settings.fetch_k, analysis.filters)

pd.DataFrame({
    "similaridade com a pergunta original": results_table(original_results)["similaridade"].head(10),
    "similaridade com o texto reescrito": results_table(results)["similaridade"].head(10),
})

In [ ]:
results_table(original_results)[["similaridade", "nota", "texto"]].head(5)

## 9. Busca híbrida, rerank e seleção das evidências

O `ReviewRetriever` repete, para cada frase de busca, os passos abaixo.

- Busca densa no Chroma, com os 40 mais próximos, e busca lexical BM25, com os 40 de maior pontuação. O BM25 encontra termos exatos, como "correios" ou "nota fiscal", e usa o mesmo filtro de metadados.
- Junta as duas listas por Reciprocal Rank Fusion (RRF): cada avaliação soma `1 / (60 + posição)` em cada lista, e quem aparece bem nas duas sobe.
- Descarta candidatos com similaridade de cosseno abaixo do mínimo configurado.
- Envia até 30 candidatos ao rerank. Um LLM barato lê a pergunta original, a frase de busca e cada avaliação, e dá uma nota de 0 a 3. Só ficam as avaliações com nota 2 ou 3. O rerank é a principal barreira contra perguntas sem base: uma avaliação sobre entrega comum recebe nota baixa numa pergunta sobre entregas por drone.

Depois, o retriever divide as vagas igualmente entre as frases de busca com resultado, para que todos os subtemas apareçam. Dentro da vaga de cada frase, o MMR (Maximal Marginal Relevance) escolhe avaliações relevantes e diferentes entre si. Textos idênticos, como dezenas de "Não recebi o produto", viram uma evidência com contador. Cada evidência recebe um ID curto, como `R1`, que o LLM usa nas citações.

A primeira tabela mostra o funil de cada frase de busca; a segunda, as evidências escolhidas.

In [ ]:
from voc_rag.lexical import LexicalIndex
from voc_rag.reranker import create_reranker
from voc_rag.retriever import ReviewRetriever

lexical = LexicalIndex(documents_df) if settings.use_bm25 else None
# Cross-encoder local por padrão (logit de relevância); RERANKER=llm usa notas de 0 a 3 do LLM.
reranker = create_reranker(settings) if settings.use_rerank else None
retriever = ReviewRetriever(store, settings, lexical, reranker)

queries = analysis.search_queries if settings.use_multi_query else analysis.search_queries[:1]
overview_question = settings.use_multi_query and analysis.question_type == "overview"
top_k = settings.overview_top_k if overview_question else settings.top_k
retrieval = retriever.retrieve(question, queries, analysis.filters, top_k)
evidences = retrieval.evidences

display(pd.DataFrame(retrieval.trace))
pd.DataFrame(
    [
        {
            "id": evidence.id,
            "busca": evidence.matched_query,
            "similaridade": evidence.similarity,
            "relevância": evidence.relevance,
            "posição densa": evidence.dense_rank,
            "posição BM25": evidence.lexical_rank,
            "repetições": evidence.occurrences,
            "nota": evidence.metadata.get("score"),
            "estado": evidence.metadata.get("state"),
            "texto": evidence.text,
        }
        for evidence in evidences
    ]
)

Se sobrarem menos de 3 evidências, o pipeline para aqui e responde que não há evidência suficiente, sem chamar o LLM de novo.

## 10. Panorama da base

O retriever escolheu as evidências pela relevância para a pergunta, com vagas divididas entre os subtemas. Com elas, você lê o que os clientes escreveram, e não consegue estimar quantos clientes escreveram algo parecido. Para os números, o `ReviewStats` usa pandas sobre as avaliações do recorte, filtradas pelas condições da etapa 7.

In [ ]:
from voc_rag.stats import filter_mask

subset = analytics[filter_mask(analytics, analysis.filters)]

print(f"Avaliações no recorte: {len(subset)}")
subset["score"].value_counts(normalize=True).sort_index().mul(100).round(1).to_frame("% das avaliações")

A classe `ReviewStats` faz a mesma filtragem e devolve o resumo em texto, no formato que o pipeline envia ao LLM.

In [ ]:
from voc_rag.stats import ReviewStats

overview = ReviewStats(analytics).describe(analysis.filters)
print(overview)

## 11. Construção do contexto

Cada evidência vira um bloco com ID, nota, categoria, situação da entrega, mês da compra, estado do cliente, a frase de busca que a encontrou e o texto. Os metadados ficam separados do texto do cliente, para que o modelo não atribua ao cliente a situação de entrega calculada pela base. O estado pertence à avaliação exibida; as repetições do mesmo texto podem vir de outros estados.

A célula imprime as mensagens com o texto que o modelo recebe.

In [ ]:
from voc_rag.generator import format_context
from voc_rag.prompts import GENERATOR_HUMAN, GENERATOR_SYSTEM

context = format_context(evidences)
generator_prompt = ChatPromptTemplate.from_messages([("system", GENERATOR_SYSTEM), ("human", GENERATOR_HUMAN)])

for message in generator_prompt.format_messages(question=question, overview=overview, context=context):
    print(f"########## {message.type} ##########")
    print(message.content)
    print()

## 12. Geração da resposta

O LLM responde no esquema `GeneratedAnswer`:

- `sufficient_evidence` indica se as avaliações respondem à pergunta;
- `summary` é a resposta direta;
- `insights` é a lista de achados, cada um com os IDs das avaliações que o sustentam;
- `limitations` diz o que as evidências não permitem afirmar.

In [ ]:
from voc_rag.schemas import GeneratedAnswer

generator_chain = generator_prompt | llm.with_structured_output(GeneratedAnswer)
answer = generator_chain.invoke({"question": question, "overview": overview, "context": context})
answer.model_dump()

## 13. Validação das citações

O prompt pede citações, e o código confere cada uma. A função `clean_citations` aceita variações como `[R1]`, `r 1` e intervalos como `R1-R3`. Depois remove IDs ausentes do contexto e descarta insights sem citação válida.

In [ ]:
from voc_rag.generator import clean_citations

valid_ids = {evidence.id for evidence in evidences}
insights_before = len(answer.insights)

for insight in answer.insights:
    cleaned_ids = clean_citations(insight.evidence_ids, valid_ids)
    removed = [raw_id for raw_id in insight.evidence_ids if raw_id.strip("[] ").upper() not in cleaned_ids]
    insight.evidence_ids = cleaned_ids
    if removed:
        print(f"Citações removidas do insight '{insight.text[:60]}': {removed}")

answer.insights = [insight for insight in answer.insights if insight.evidence_ids]
print(f"Insights mantidos: {len(answer.insights)} de {insights_before}")

## 14. Resultado final

A decisão segue a ordem do pipeline. Pergunta fora do escopo gera recusa. Poucas evidências, `sufficient_evidence` falso ou zero insights válidos geram a mensagem de evidência insuficiente. Nos demais casos, a resposta sai com os insights e a tabela de evidências.

Neste notebook, as 13 etapas rodaram em sequência para você ver cada saída. A classe `VoCPipeline` interrompe o fluxo no primeiro motivo de recusa e economiza chamadas ao LLM.

In [ ]:
from voc_rag.prompts import INSUFFICIENT_MESSAGE, OUT_OF_SCOPE_MESSAGE
from voc_rag.schemas import PipelineResult, Status

if not analysis.in_scope:
    result = PipelineResult(question, Status.OUT_OF_SCOPE, OUT_OF_SCOPE_MESSAGE, analysis=analysis)
elif len(evidences) < settings.min_evidences or not answer.sufficient_evidence or not answer.insights:
    result = PipelineResult(
        question, Status.INSUFFICIENT, INSUFFICIENT_MESSAGE,
        limitations=answer.limitations, evidences=evidences, analysis=analysis, overview=overview,
    )
else:
    result = PipelineResult(
        question, Status.ANSWERED, answer.summary,
        insights=answer.insights, limitations=answer.limitations,
        evidences=evidences, analysis=analysis, overview=overview,
    )

Markdown(result.to_markdown())

## 15. Mesmo fluxo com as classes do projeto

`build_pipeline` monta as peças das etapas anteriores. O `QueryAnalyzer` cobre a etapa 6, o `ReviewRetriever` cobre as etapas 7 a 9, o `ReviewStats` cobre a etapa 10 e o `AnswerGenerator` cobre as etapas 11 a 13. O `VoCPipeline.ask` executa o fluxo em uma chamada e guarda em `trace` o funil, os tempos e os tokens. O LLM varia o texto a cada chamada, então a resposta pode diferir da etapa 14.

In [ ]:
from voc_rag.pipeline import build_pipeline

pipeline = build_pipeline(settings)
result = pipeline.ask(question)
print(f"Tempo: {result.trace['timings_s']} | Tokens: {result.trace['tokens']}")
Markdown(result.to_markdown())

## 16. Recusas e calibração do corte de similaridade

A primeira pergunta abaixo está fora do escopo, e o pipeline deve recusá-la antes da busca. A segunda está no escopo, mas a base não tem avaliações sobre o assunto. O corte de similaridade, o rerank ou a checagem do LLM deve barrar essa pergunta. O campo `stopped_at` do rastreio diz qual barreira recusou.

In [ ]:
refusal_questions = [
    "Qual é a previsão do tempo para São Paulo amanhã?",
    "O que os clientes dizem sobre o programa de cashback em criptomoedas da loja?",
]
for refusal_question in refusal_questions:
    refusal = pipeline.ask(refusal_question)
    print(f"Barreira: {refusal.trace.get('stopped_at')}")
    display(Markdown(refusal.to_markdown()))

O corte de 0,40 é o nosso ponto de partida. Para calibrarmos, devemos comparar a maior similaridade encontrada em perguntas que a base consegue responder com a de perguntas que ela não consegue. Um bom corte fica entre os dois grupos. Se os grupos se misturarem, o rerank e a checagem `sufficient_evidence` do LLM continuam como barreiras. O comando `python -m voc_rag.cli evaluate --split dev` faz a mesma comparação com mais perguntas e grava o resultado em `docs/avaliacao.md`.

In [ ]:
calibration_questions = {
    "a base responde": [
        "O que os clientes relatam sobre problemas relacionados à entrega?",
        "Quais aspectos da experiência de compra são mais elogiados pelos clientes?",
        "Os clientes reclamam de produtos que chegaram quebrados?",
    ],
    "a base não responde": [
        "O que os clientes dizem sobre o programa de cashback em criptomoedas da loja?",
        "Como os clientes avaliam o aplicativo de realidade virtual da loja?",
    ],
}

rows = []
for group, group_questions in calibration_questions.items():
    for calibration_question in group_questions:
        calibration_analysis = pipeline.analyzer.analyze(calibration_question)
        search_query = calibration_analysis.search_queries[0]
        top_results = search_with_similarity(store, search_query, 5)
        similarities = [similarity for _, _, similarity in top_results]
        rows.append({
            "grupo": group,
            "pergunta": calibration_question,
            "primeira frase de busca": search_query,
            "maior similaridade": round(max(similarities), 3),
            "média das 5 maiores": round(sum(similarities) / len(similarities), 3),
        })

pd.DataFrame(rows)

Para mudarmos o corte, devemos alterar `MIN_SIMILARITY` no `.env` e reiniciar o notebook.